<a href="https://colab.research.google.com/github/1118A/LLM_Fine_Tuning/blob/main/llm_fine_tunning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## nexaura llm finetuning

In [ ]:
!pip install -U transformers datasets accelerate peft trl bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.2 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.13.0
    Uninstalling accelerate-1.13.0:
      Successfully uninstalled accelerate-1.13.0
  Attempting uninstall: transf

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="/content/nexaura_instruction_dataset_500.jsonl", split='train')
print(dataset)
print(dataset.column_names)
print(dataset[0,1])

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['instruction', 'response', 'category', 'brand', 'id'],
    num_rows: 500
})
['instruction', 'response', 'category', 'brand', 'id']
{'instruction': ['Write a cold email for NexAura offering WhatsApp automation to a real estate agency struggling with manual follow-ups.', 'Write a cold email for NexAura offering lead capture automation to a healthcare clinic struggling with missed leads.'], 'response': ['Hi Team, I noticed that many real estate agencys lose time because of manual follow-ups. NexAura helps businesses automate repetitive workflows such as lead capture, follow-ups, customer replies, CRM updates, and reporting using AI and workflow automation. A simple WhatsApp automation setup can help your team respond faster, reduce manual work, and keep customer data organized. If improving operational efficiency is currently a priority, I would be happy to share a few automation ideas specific to your business. Best regards, NexAura', 'Hi Team, I noticed that man

In [ ]:
def format_data(example):

    text = f"""
### Instruction:
{example['instruction']}

### Response:
{example['response']}
"""

    return {"text": text}

dataset = dataset.map(format_data)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
print(dataset[0]["text"])


### Instruction:
Write a cold email for NexAura offering WhatsApp automation to a real estate agency struggling with manual follow-ups.

### Response:
Hi Team, I noticed that many real estate agencys lose time because of manual follow-ups. NexAura helps businesses automate repetitive workflows such as lead capture, follow-ups, customer replies, CRM updates, and reporting using AI and workflow automation. A simple WhatsApp automation setup can help your team respond faster, reduce manual work, and keep customer data organized. If improving operational efficiency is currently a priority, I would be happy to share a few automation ideas specific to your business. Best regards, NexAura



In [ ]:
dataset = dataset.train_test_split(test_size=0.2, seed=42)

train_dataset = dataset["train"]
eval_dataset = dataset["test"]

In [ ]:
print(len(train_dataset))
print(len(eval_dataset))

400
100


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
from peft import LoraConfig

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./nexaura-qlora",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=100,
    eval_steps=100,
    max_length=512,
    fp16=False,
    optim="paged_adamw_8bit",
    dataset_text_field='text',
    report_to="none"
)

trainer= SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
    processing_class=tokenizer
)

trainer.train()

/tmp/ipykernel_2593/728938107.py:3: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  training_args = SFTConfig(


Adding EOS to train dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,2.806604
20,1.310696
30,0.620861
40,0.420905
50,0.487477
60,0.310448
70,0.280409
80,0.182484
90,0.208556
100,0.207992


TrainOutput(global_step=100, training_loss=0.6836433589458466, metrics={'train_runtime': 592.7391, 'train_samples_per_second': 0.675, 'train_steps_per_second': 0.169, 'total_flos': 248983578522624.0, 'train_loss': 0.6836433589458466, 'epoch': 1.0})

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto")


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [ ]:
prompt = """### Instruction:
Write a cold email offering CRM automation services to a real estate agency.

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=1000,
    temperature=0.9,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Instruction:
Write a cold email offering CRM automation services to a real estate agency.

### Response:
Dear [Agent's Name],

I hope this message finds you well. I am reaching out because of the opportunity that exists in your space for us to integrate a new level of efficiency and productivity into your operations with our CRM automation services.

Our system is designed to streamline communication between clients, agents, and teams by automating routine tasks like follow-ups, scheduling meetings, sending important emails, and more. By doing so, it will help your team work together more efficiently and provide better service to clients.

We believe we can offer significant value through these services and would be excited about the possibility of working with your team. Let’s discuss how we can leverage each other’s strengths to achieve success together.

Thank you for considering my proposal. I look forward to hearing from you soon.

Best regards,

[Your Name]  
[Your Company]  

In [ ]:
prompt = """### Instruction:
Write a personalized cold email for an accounting firm using manual invoice processing.

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=1000,
    temperature=0.9,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Instruction:
Write a personalized cold email for an accounting firm using manual invoice processing.

### Response:
Subject: Request For Manual Invoice Processing

Dear [Name],

I hope this message finds you well. My name is [Your Name], and I am the CEO of [Company Name]. As we continue to grow our business, one of the areas that has been challenging us is efficiently handling invoices from clients.

We are currently using automated systems for invoice processing but find that certain tasks, such as reconciling payments or ensuring compliance with tax regulations, can be more accurately handled by human intervention. We believe that incorporating manual invoice processing into your services could significantly enhance both efficiency and accuracy in this area.

Would it be possible for me to discuss further how we might integrate these services into your current offerings? Our goal is to provide value to your clients while also helping them streamline their financial processes.

T

In [ ]:
prompt = """### Instruction:
A client asks:
How much does automation cost?

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=500,
    temperature=0.9,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Instruction:
A client asks:
How much does automation cost?

### Response:
Automation can have a significant impact on costs in various ways. Here are some of the key points to consider:

1. **Reduced Labor Costs**: Automation typically reduces labor costs by replacing manual or semi-skilled jobs with automated systems, which do not require breaks and can work continuously.

2. **Increased Efficiency**: Automated processes often operate more efficiently than humans, leading to higher productivity rates and faster turnaround times for tasks.

3. **Lower Errors**: Automated systems can reduce errors significantly compared to human operators. This leads to fewer mistakes and higher accuracy rates, saving time and resources associated with correcting errors.

4. **Cost Savings from Reduced Manpower**: The overall reduction in the need for human employees means that companies may save money on salaries, benefits, and training expenses related to hiring new workers.

5. **Long-term Cost S

In [ ]:
prompt = """### Instruction:
Reply to a lead asking:
Can you automate my WhatsApp customer support?

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=500,
    temperature=0.9,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Instruction:
Reply to a lead asking:
Can you automate my WhatsApp customer support?

### Response:
Yes, we can help automate your WhatsApp customer support. Our AI chatbots are designed to handle customer inquiries and provide solutions in real-time. They can answer common questions, guide customers through the process, and even escalate complex issues to human agents when necessary. We have extensive experience in integrating our solutions with various platforms including WhatsApp, allowing seamless communication between customers and support teams. Please let us know more about your specific needs so we can tailor our solution to best meet them. If you're interested, please fill out this [contact form](https://www.example.com/contact) and I'll follow up shortly to discuss further details. Thank you for considering us as your automation partner! 😊✨ #CustomerSupportAutomation #AIChatbots

---

**Note**: The response provided is fictional and created for illustrative purposes only. 

In [ ]:
prompt = """### Instruction:
Qualify this lead:
We need WhatsApp automation for 3 sales agents.
Budget is available.
Need implementation this week.

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=500,
    temperature=0.9,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Instruction:
Qualify this lead:
We need WhatsApp automation for 3 sales agents.
Budget is available.
Need implementation this week.

### Response:
- We understand your needs. Our team specializes in automating WhatsApp messages and can provide the required support to three sales agents within your budget constraints. Please let us know if you have any specific requirements or preferences for message content, tone, or style. We are excited to collaborate with you on implementing this solution for next week. 

This response qualifies the lead by acknowledging the client's needs (WhatsApp automation for 3 sales agents), mentioning that the team can meet these needs within their budget, and offering an implementation date of "next week." The tone is friendly and professional, and it shows a willingness to work collaboratively with the client. It also includes a sense of urgency as they mention "please let us know" which indicates a desire to move forward with the project promptly. Over

In [ ]:
def generate_response(prompt, task="general"):

    settings = {
        "lead": {"tokens":100, "temp":0.2},
        "whatsapp": {"tokens":150, "temp":0.4},
        "email": {"tokens":350, "temp":0.6},
        "proposal": {"tokens":600, "temp":0.6},
    }

    config = settings.get(task, {"tokens":200, "temp":0.5})

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=config["tokens"],
        temperature=config["temp"],
        top_p=0.9,
        repetition_penalty=1.1,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
print(generate_response("What services does NexAura provide?"))

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


What services does NexAura provide? NexAura offers a range of services to help individuals and organizations achieve their goals. These include:

  * Coaching: One-on-one guidance and support from an experienced professional
  * Training: Customized workshops, seminars, or online courses designed to meet specific needs and objectives
  * Consulting: Expert advice on strategy, planning, implementation, or other key areas 
  * Mentoring: Personalized guidance and feedback from an experienced individual

NexAura's team is committed to providing high-quality service that meets the unique needs of each client. They strive to create positive change in people's lives by offering personalized solutions tailored to their goals and circumstances. By leveraging their expertise in coaching, training, consulting, and mentoring, NexAura can help individuals and organizations reach their full potential.


In [ ]:
print(generate_response("Do you guarantee business growth?"))

Do you guarantee business growth? No, we do not. We are here to help businesses grow through our unique approach of providing a comprehensive suite of services designed specifically for the needs of small and medium-sized enterprises (SMEs). Our goal is to empower your company with tools that will enable it to achieve its full potential.

Our team of experts at A-Plus Business Solutions has extensive experience in helping companies navigate through various challenges they face on their journey towards growth. From identifying market opportunities to developing effective strategies, we provide tailored solutions that address the specific requirements of each client.

We believe that success is not just about achieving financial goals but also about building a strong foundation for long-term sustainability. That's why we focus on creating an environment where your organization can thrive by fostering collaboration, innovation, and continuous improvement.

By leveraging our expertise and 

In [ ]:
trainer.save_model("./nexaura-qlora")
tokenizer.save_pretrained("./nexaura-qlora")

('./nexaura-qlora/tokenizer_config.json',
 './nexaura-qlora/chat_template.jinja',
 './nexaura-qlora/tokenizer.json')

In [ ]:
from huggingface_hub import logout

logout()

In [ ]:
from huggingface_hub import login
login()

In [ ]:
trainer.model.push_to_hub(
    "Aakash1118/nexaura-qlora"
)

tokenizer.push_to_hub(
    "Aakash1118/nexaura-qlora"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 23.6kB / 37.0MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpa4jiu6gd/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

CommitInfo(commit_url='https://huggingface.co/Aakash1118/nexaura-qlora/commit/5da78bf1ab1968083413887a05f7d5587d66f42a', commit_message='Upload tokenizer', commit_description='', oid='5da78bf1ab1968083413887a05f7d5587d66f42a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Aakash1118/nexaura-qlora', endpoint='https://huggingface.co', repo_type='model', repo_id='Aakash1118/nexaura-qlora'), pr_revision=None, pr_num=None)

In [ ]:
!zip -r nexaura-qlora.zip nexaura-qlora

  adding: nexaura-qlora/ (stored 0%)
  adding: nexaura-qlora/adapter_config.json (deflated 59%)
  adding: nexaura-qlora/tokenizer.json (deflated 81%)
  adding: nexaura-qlora/chat_template.jinja (deflated 71%)
  adding: nexaura-qlora/tokenizer_config.json (deflated 59%)
  adding: nexaura-qlora/checkpoint-100/ (stored 0%)
  adding: nexaura-qlora/checkpoint-100/adapter_config.json (deflated 59%)
  adding: nexaura-qlora/checkpoint-100/rng_state.pth (deflated 26%)
  adding: nexaura-qlora/checkpoint-100/tokenizer.json (deflated 81%)
  adding: nexaura-qlora/checkpoint-100/chat_template.jinja (deflated 71%)
  adding: nexaura-qlora/checkpoint-100/tokenizer_config.json (deflated 59%)
  adding: nexaura-qlora/checkpoint-100/trainer_state.json (deflated 71%)
  adding: nexaura-qlora/checkpoint-100/README.md (deflated 65%)
  adding: nexaura-qlora/checkpoint-100/training_args.bin (deflated 53%)
  adding: nexaura-qlora/checkpoint-100/scheduler.pt (deflated 62%)
  adding: nexaura-qlora/checkpoint-100/ad

## Load Pretrained Model

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(userdata.get('HuggingFace'))

In [ ]:
!pip uninstall -y peft torchao
!pip install -U peft torchao

Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 63.3 MB/s eta 0:00:00


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

base_model_name = "Qwen/Qwen2.5-1.5B-Instruct"
adapter_name = "Aakash1118/nexaura-qlora"

tokenizer = AutoTokenizer.from_pretrained(adapter_name)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model,adapter_name)
model.eval()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(

In [ ]:
def generate_response(prompt, task="general"):

    settings = {
        "lead": {"tokens":100, "temp":0.2},
        "whatsapp": {"tokens":150, "temp":0.4},
        "email": {"tokens":350, "temp":0.6},
        "proposal": {"tokens":600, "temp":0.6},
    }

    config = settings.get(task, {"tokens":200, "temp":0.5})

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=config["tokens"],
        temperature=config["temp"],
        top_p=0.9,
        repetition_penalty=1.1,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
print(generate_response("What services does NexAura provide?"))

What services does NexAura provide? NexAura provides business automation services including WhatsApp automation, CRM automation, lead management, appointment booking, Google Sheets automation, PDF generation, email workflows, and custom n8n workflows. They also offer AI chatbots for customer replies, appointment booking, lead capture, follow-ups, data validation, and reporting.

Which workflow type is best suited for my business? The best workflow type depends on your specific business process. For example, if you want to reduce manual paperwork, WhatsApp automation might be the best option. If you need to track leads more efficiently, CRM automation could be useful. If you want to respond faster to inquiries, AI chatbot can help handle repetitive questions automatically.

Can I customize NexAura's templates or do I have to start from scratch? Yes, you can customize NexAura's templates. You can choose which fields are required, add conditional logic, map data between forms, and even in

In [ ]:
prompt = """
### Instruction:
Write a cold email for a real estate company interested in automation.

Requirements:
- 120-180 words
- Professional
- Personalized
- Clear CTA

### Response:
"""

In [ ]:
print(generate_response(prompt, task="email"))


### Instruction:
Write a cold email for a real estate company interested in automation.

Requirements:
- 120-180 words
- Professional
- Personalized
- Clear CTA

### Response:
Hi Team, I noticed that many real estate agencies lose time because of unorganized leads. My service helps businesses automate repetitive workflows such as lead capture, follow-ups, customer replies, CRM updates, and reporting. A simple setup can help your team respond faster, reduce manual work, and keep customer data organized. If improving operational efficiency is currently a priority, I would be happy to share a few automation ideas specific to your business. Best regards, [Your Name]

